# 16 — Conformal and selective baselines

**Objective.** Benchmark conformal singleton/set size, confidence, margin, entropy, epistemic disagreement, density, and predicted explanation loss under deployable 2009 thresholds and retrospective matched coverage.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Singleton status is treated only as an empirical baseline; it is not an individual correctness or explanation-reliability certificate.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("16", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json, joblib
import numpy as np
import pandas as pd
from cruxvc.io import read_table, write_table
from cruxvc.manifest import append_test_access_log
from cruxvc.workflow import fit_refit_calibrated_model
from cruxvc.selective import (
    cross_fitted_mondrian_binary_sets,
    cross_fitted_split_conformal_binary_sets,
    deployable_gate_evaluation,
    mondrian_binary_sets,
    retrospective_matched_coverage_evaluation,
    split_conformal_binary_sets,
    standard_gate_scores,
    summarize_risk_coverage,
)

append_test_access_log(P, stage_id="16", purpose="locked uncertainty-gating benchmark", resources=[P.predictions / "final_predictions.parquet", P.processed / "cohort_labels.parquet"])
features = read_table(P.processed / "features_strict.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
splits = read_table(P.protocol / "split_ids.parquet")
registry = read_table(P.models / "calibration_registry.parquet")
final_predictions = read_table(P.predictions / "final_predictions.parquet")
data = features.merge(cohort, on=["case_id", "company_permalink", "t0"]).merge(splits[["case_id", "time_block"]], on="case_id")
development = data[data["time_block"].astype(str).eq("development")].copy()
cal2008 = data[data["time_block"].astype(str).eq("probability_calibration")].copy()
risk2009 = data[data["time_block"].astype(str).eq("risk_calibration")].copy()
test = data[data["time_block"].astype(str).eq("final_test")].copy()
seed_table = pd.read_csv(P.protocol / "seed_registry.csv")
primary = registry[(registry["analysis_role"].eq("matched_reference")) & registry["outcome"].eq("F36")].sort_values("platt_oof_log_loss").iloc[0]
primary_model = joblib.load(primary["calibrated_model_path"])
p2009 = primary_model.predict_proba(risk2009)[:, 1]
ptest = primary_model.predict_proba(test)[:, 1]


In [ ]:
feature_columns = [c for c in features.columns if c not in {"case_id", "company_permalink", "t0"}]
# Distinguish same-family bootstrap epistemic variance from cross-family
# selective-ensemble disagreement. Every refit is trained only on 2005–2007
# and probability-calibrated on 2008 before any 2009 label is opened.
refit_count = int(PROFILE["esrc_reference_panel_refits"])
refit_seeds = (
    seed_table[seed_table["purpose"].eq("bootstrap_refit")]
    .sort_values("index")["seed"]
    .head(refit_count)
    .astype(int)
    .tolist()
)
primary_parameters = json.loads(primary["parameters_json"])
epistemic_models = [
    fit_refit_calibrated_model(
        development,
        cal2008,
        feature_columns=feature_columns,
        outcome="F36",
        family=str(primary["family"]),
        parameters=primary_parameters,
        seed=seed,
    )
    for seed in refit_seeds
]
epistemic_2009 = np.column_stack([model.predict_proba(risk2009)[:, 1] for model in epistemic_models])
epistemic_test = np.column_stack([model.predict_proba(test)[:, 1] for model in epistemic_models])
family_jobs = registry[(registry["analysis_role"].eq("matched_reference")) & registry["outcome"].eq("F36")]
family_ensemble_2009 = np.column_stack([joblib.load(path).predict_proba(risk2009)[:, 1] for path in family_jobs["calibrated_model_path"]])
family_ensemble_test = np.column_stack([joblib.load(path).predict_proba(test)[:, 1] for path in family_jobs["calibrated_model_path"]])
density_model = joblib.load(P.models / "development_density_model.joblib")
meta_model = joblib.load(P.models / "predicted_explanation_loss_meta_model.joblib")
scores2009 = standard_gate_scores(
    p2009,
    epistemic_variance=epistemic_2009.var(axis=1),
    model_disagreement=(family_ensemble_2009 >= 0.5).std(axis=1),
    density_score=-density_model.decision_function(risk2009[feature_columns]),
    predicted_explanation_loss=meta_model.predict(risk2009[feature_columns]),
)
scorestest = standard_gate_scores(
    ptest,
    epistemic_variance=epistemic_test.var(axis=1),
    model_disagreement=(family_ensemble_test >= 0.5).std(axis=1),
    density_score=-density_model.decision_function(test[feature_columns]),
    predicted_explanation_loss=meta_model.predict(test[feature_columns]),
)
standard_cp = split_conformal_binary_sets(p2009, risk2009["F36"], ptest, alpha=0.10)
mondrian_cp = mondrian_binary_sets(p2009, risk2009["F36"], ptest, alpha=0.10)
scorestest["conformal_set_size"] = standard_cp["set_size"].to_numpy(dtype=float)
scorestest["mondrian_set_size"] = mondrian_cp["set_size"].to_numpy(dtype=float)
# A 2009 case's gate score must not use that same case's label. Produce each
# calibration-block set size from thresholds fitted on the other folds.
crossfit_seed = int(CFG["execution"]["random_seed"]) + 160
cp2009 = cross_fitted_split_conformal_binary_sets(
    p2009, risk2009["F36"], alpha=0.10, folds=5, seed=crossfit_seed
)
mcp2009 = cross_fitted_mondrian_binary_sets(
    p2009, risk2009["F36"], alpha=0.10, folds=5, seed=crossfit_seed
)
scores2009["conformal_set_size"] = cp2009["set_size"].to_numpy(dtype=float)
scores2009["mondrian_set_size"] = mcp2009["set_size"].to_numpy(dtype=float)


In [ ]:
# Explanation-risk population is the frozen 300-case audit sample; prediction-only curves remain available on all 788.
audit_ids = set(pd.read_csv(P.protocol / "local_audit_case_ids.csv")["case_id"])
common_mask = test["case_id"].isin(audit_ids).to_numpy()
common_test = test.loc[common_mask].reset_index(drop=True)
common_scores = scorestest.loc[common_mask].reset_index(drop=True)
y = common_test["F36"].to_numpy(dtype=int)
p = ptest[common_mask]
prediction_error = ((p >= 0.5).astype(int) != y).astype(float)
logloss = -(y * np.log(np.clip(p, 1e-6, 1 - 1e-6)) + (1 - y) * np.log(np.clip(1 - p, 1e-6, 1 - 1e-6)))

cross = read_table(P.inference / "rq1_cross_spec_distances.parquet")
construct_loss = cross.groupby("case_id")["sqrt_jsd"].mean().reindex(common_test["case_id"]).fillna(1.0).to_numpy()
within = read_table(P.inference / "rq1_within_refit_distances.parquet")
instability = within[within["outcome"].eq("F36")].groupby("case_id")["sqrt_jsd"].mean().reindex(common_test["case_id"]).fillna(1.0).to_numpy()
faithfulness = read_table(P.controls / "rq3_faithfulness_case_summary.parquet").set_index("case_id")
faithfulness_loss = (1 - np.clip(faithfulness["normalized_top_minus_random"], 0, 1)).reindex(common_test["case_id"]).fillna(1.0).to_numpy()
losses = pd.DataFrame({
    "prediction_error": prediction_error,
    "log_loss": logloss,
    "explanation_instability": instability,
    "construct_fragility": construct_loss,
    "faithfulness_proxy_loss": faithfulness_loss,
})

In [ ]:
coverages = CFG["metrics"]["gate_coverages"]
deployable = deployable_gate_evaluation(scores2009, common_scores, losses, coverages)
retrospective_scores = common_scores.copy()
retrospective_scores["random_acceptance"] = np.random.default_rng(int(CFG["execution"]["random_seed"])).random(len(common_scores))
retrospective_scores["oracle_explanation_loss"] = construct_loss + instability + faithfulness_loss
retrospective = retrospective_matched_coverage_evaluation(retrospective_scores, losses, coverages)
deployable_summary = summarize_risk_coverage(deployable)
retrospective_summary = summarize_risk_coverage(retrospective)

# Prediction fields are provenance for L_Y in Notebook 17, not candidate gates.
scores2009.insert(0, "case_id", risk2009["case_id"].to_numpy())
scores2009["probability"] = p2009
scores2009["predicted_class"] = (p2009 >= 0.5).astype("int8")
common_scores.insert(0, "case_id", common_test["case_id"].to_numpy())
common_scores["probability"] = p
common_scores["predicted_class"] = (p >= 0.5).astype("int8")
scores_2009_path = write_table(scores2009, P.selective / "risk_calibration_gate_scores_2009.parquet")
scores_test_path = write_table(common_scores, P.selective / "test_gate_scores_common_population.parquet")
losses_path = write_table(pd.concat([common_test[["case_id"]], losses], axis=1), P.selective / "test_losses_common_population.parquet")
deployable_path = write_table(deployable, P.selective / "deployable_gate_curves.csv")
retrospective_path = write_table(retrospective, P.selective / "retrospective_matched_coverage_curves.csv")
summary_path = write_table(pd.concat([deployable_summary.assign(evaluation="deployable_2009_threshold"), retrospective_summary.assign(evaluation="retrospective_identical_coverage")]), P.selective / "gate_partial_aurc_summary.csv")
conformal_path = write_table(pd.concat([test[["case_id"]].reset_index(drop=True), standard_cp.add_prefix("standard_"), mondrian_cp.add_prefix("mondrian_")], axis=1), P.selective / "conformal_set_outputs_2010.parquet")
CTX.recorder.complete([scores_2009_path, scores_test_path, losses_path, deployable_path, retrospective_path, summary_path, conformal_path])
print(pd.read_csv(summary_path).sort_values(["loss", "normalized_partial_aurc"]).to_string(index=False))
